# ⚡ Intel ARC GPU 실행 — 환자안전 모니터링

**Intel Extension for PyTorch (IPEX)** 를 사용해 Intel ARC / Iris Xe GPU에서
Fine-tuned MobileVLM V2를 실행합니다.

| 항목 | 내용 |
|---|---|
| 백엔드 | Intel Extension for PyTorch (XPU) |
| 대상 하드웨어 | Intel ARC / Iris Xe GPU |
| 예상 속도 | 3~8초/장 (ARC 기준) |
| dtype | float16 (ARC) / float32 (Xe) |

> ⚠️ **사전 요구사항**: Intel ARC 드라이버 설치 필요  
> 드라이버: https://www.intel.com/content/www/us/en/download/785597/intel-arc-iris-xe-graphics-windows.html

## 📦 Cell 1: IPEX 설치 (최초 1회)

In [ ]:
import subprocess

# 기본 패키지
base_pkgs = ['transformers', 'peft', 'sentencepiece', 'timm',
             'torchvision', 'pillow', 'opencv-python', 'einops',
             'accelerate', 'safetensors']
for pkg in base_pkgs:
    subprocess.run(['pip', 'install', '-q', pkg])
    print(f'✅ {pkg}')

# Intel Extension for PyTorch (XPU 지원 버전)
# torch 버전에 맞는 IPEX 설치
print('\nIPEX 설치 중 (시간 소요)...')
subprocess.run([
    'pip', 'install', '-q',
    'intel-extension-for-pytorch==2.1.30+xpu',
    '--extra-index-url', 'https://pytorch-extension.intel.com/release-whl/stable/xpu/us/'
])
print('✅ IPEX 설치 완료')
print('\n⚠️ 커널 재시작 후 Cell 2부터 실행하세요!')

## ⚙️ Cell 2: GPU 확인 & 경로 설정

In [2]:
import os, sys, zipfile, time, cv2
import torch
from datetime import datetime
from pathlib import Path
from PIL import Image

# ── Intel ARC XPU 확인 ────────────────────────────────────────
try:
    import intel_extension_for_pytorch as ipex
    xpu_available = torch.xpu.is_available()
    print(f'✅ IPEX 로드 성공')
    print(f'   XPU 사용 가능: {xpu_available}')
    if xpu_available:
        print(f'   GPU 이름: {torch.xpu.get_device_name(0)}')
        DEVICE = 'xpu'
        DTYPE  = torch.float16
    else:
        print('⚠️ XPU 없음 → CPU 폴백')
        DEVICE = 'cpu'
        DTYPE  = torch.float32
except ImportError:
    print('❌ IPEX 미설치 → Cell 1 먼저 실행하세요')
    DEVICE = 'cpu'
    DTYPE  = torch.float32

# ── 경로 설정 ─────────────────────────────────────────────────
BASE_DIR      = Path(r'c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM')
ADAPTER_DIR   = BASE_DIR / 'mobilevlm_lora_adapter'
MERGED_DIR    = BASE_DIR / 'mobilevlm_merged'          # LoRA 병합 모델 저장 위치
VIDEO_DIR     = BASE_DIR / 'demo_videos'
RESULT_DIR    = BASE_DIR / 'demo_results'
MOBILEVLM_DIR = BASE_DIR / 'MobileVLM'

# ── 추론 설정 ─────────────────────────────────────────────────
INTERVAL_SEC  = 2
PROMPT        = '의료진에게 환자의 현재 상태를 보고하세요.'
ALERT_KEYWORD = '낙상'
MODEL_NAME    = 'mtgv/MobileVLM_V2-1.7B'

RESULT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(MOBILEVLM_DIR))

print(f'\n실행 디바이스: {DEVICE} ({DTYPE})')

❌ IPEX 미설치 → Cell 1 먼저 실행하세요

실행 디바이스: cpu (torch.float32)


## 🔗 Cell 3: LoRA 병합 (Merge) — 최초 1회만 실행

In [4]:
# LoRA를 base 모델에 병합 → 단일 모델 파일로 저장
# 병합된 모델은 MERGED_DIR에 저장되어 이후 재사용

if MERGED_DIR.exists():
    print(f'✅ 병합 모델 이미 존재: {MERGED_DIR}')
    print('   Cell 4로 바로 이동하세요.')
else:
    print('LoRA 병합 시작 (시간 소요: 3~5분)...')
    from peft import PeftModel
    from mobilevlm.model.mobilevlm import load_pretrained_model
    from mobilevlm.utils import disable_torch_init

    disable_torch_init()
    tokenizer, model_base, image_processor, _ = load_pretrained_model(
        MODEL_NAME, load_8bit=False, load_4bit=False, device='cpu', device_map='cpu'
    )

    print('LoRA 어댑터 로드 중...')
    peft_model = PeftModel.from_pretrained(model_base, str(ADAPTER_DIR.resolve()))
    peft_model = peft_model.float()

    print('LoRA 병합 중... (merge_and_unload)')
    merged_model = peft_model.merge_and_unload()   # ← LoRA → base에 통합

    print(f'병합 모델 저장 중: {MERGED_DIR}')
    MERGED_DIR.mkdir(exist_ok=True)
    merged_model.save_pretrained(str(MERGED_DIR))
    tokenizer.save_pretrained(str(MERGED_DIR))
    print('✅ LoRA 병합 완료!')
    print(f'   저장 위치: {MERGED_DIR}')

LoRA 병합 시작 (시간 소요: 3~5분)...


c:\Users\ASUS\anaconda3\envs\ds_study\lib\site-packages\torch\nn\modules\module.py:2068: UserWarning: for vision_model.embeddings.class_embedding: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(f'for {key}: copying from a non-meta parameter in the checkpoint to a meta '
c:\Users\ASUS\anaconda3\envs\ds_study\lib\site-packages\torch\nn\modules\module.py:2068: UserWarning: for vision_model.embeddings.patch_embedding.weight: copying from a non-meta parameter in the checkpoint to a meta parameter in the current model, which is a no-op. (Did you mean to pass `assign=True` to assign items in the state dictionary to their corresponding key in the module instead of copying them in place?)
  warnings.warn(f'for {key}: copying from a non-meta parameter in the checkpoi

LoRA 어댑터 로드 중...
LoRA 병합 중... (merge_and_unload)
병합 모델 저장 중: c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM\mobilevlm_merged
✅ LoRA 병합 완료!
   저장 위치: c:\Users\ASUS\Desktop\제로베이스\Patient_Behavior_Reporting_System-VLM\mobilevlm_merged


## 🤖 Cell 4: 병합 모델 로드 → ARC GPU(XPU)로 이동

In [ ]:
import torch
from mobilevlm.model.mobilevlm import load_pretrained_model
from mobilevlm.utils import disable_torch_init

# 병합된 단일 모델 로드 (LoRA 없이)
print(f'병합 모델 로드 중: {MERGED_DIR}')
disable_torch_init()

tokenizer, model, image_processor, _ = load_pretrained_model(
    str(MERGED_DIR),
    load_8bit=False,
    load_4bit=False,
    device='cpu',    # 일단 CPU에 로드 후 XPU로 이동
)

# XPU(Intel ARC)로 모델 이동
if DEVICE == 'xpu':
    print('모델을 Intel ARC XPU로 이동 중...')
    model = model.to(DEVICE)
    # IPEX 최적화 적용 (추론 전용)
    model = ipex.optimize(model, dtype=DTYPE, inplace=True)
    print(f'✅ Intel ARC XPU 최적화 완료')
else:
    model = model.float()
    print('✅ CPU 모드로 로드 완료')

model.eval()
print(f'\n모델 디바이스: {next(model.parameters()).device}')

## ⚙️ Cell 5: 추론 함수 정의

In [ ]:
from mobilevlm.utils import process_images, tokenizer_image_token
from mobilevlm.constants import IMAGE_TOKEN_INDEX, DEFAULT_IMAGE_TOKEN
from mobilevlm.conversation import conv_templates

def infer_frame(pil_image):
    """PIL 이미지 1장 → ARC GPU 추론 → 결과 문자열"""
    # 이미지 전처리
    image_tensor = process_images([pil_image], image_processor, model.config)[0]
    image_tensor = image_tensor.to(DEVICE, dtype=DTYPE)

    # 프롬프트 구성
    full_prompt = DEFAULT_IMAGE_TOKEN + '\n' + PROMPT
    conv = conv_templates['v1'].copy()
    conv.append_message(conv.roles[0], full_prompt)
    conv.append_message(conv.roles[1], None)

    input_ids = tokenizer_image_token(
        conv.get_prompt(), tokenizer, IMAGE_TOKEN_INDEX, return_tensors='pt'
    ).unsqueeze(0).to(DEVICE)

    with torch.inference_mode():
        with torch.autocast(DEVICE, dtype=DTYPE, enabled=(DEVICE == 'xpu')):
            output_ids = model.generate(
                input_ids,
                images=image_tensor.unsqueeze(0),
                do_sample=False,
                num_beams=3,
                min_new_tokens=15,
                max_new_tokens=128,
                repetition_penalty=1.3,
                use_cache=True,
            )
    return tokenizer.decode(
        output_ids[0, input_ids.shape[1]:], skip_special_tokens=True
    ).strip()

def format_time(seconds):
    return f'{int(seconds)//60:02d}:{int(seconds)%60:02d}'

# 속도 테스트
print('추론 속도 테스트 중 (빈 이미지)...')
test_img = Image.new('RGB', (336, 336), color=(128, 128, 128))
t0 = time.time()
result = infer_frame(test_img)
print(f'✅ 테스트 완료: {time.time()-t0:.1f}초')
print(f'   출력: {result}')

## 🎬 Cell 6: 영상 추론 실행

In [ ]:
video_files = sorted([v for v in VIDEO_DIR.iterdir()
                      if v.suffix.lower() in ('.mp4', '.avi', '.mov', '.mkv')])

if not video_files:
    print('[ERROR] demo_videos/ 폴더에 영상을 넣어주세요.')
else:
    VIDEO_INDEX = 0   # ← 처리할 영상 번호
    video_path  = video_files[VIDEO_INDEX]

    cap        = cv2.VideoCapture(str(video_path))
    fps        = cap.get(cv2.CAP_PROP_FPS) or 30
    total_f    = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_step = max(1, int(fps * INTERVAL_SEC))

    print(f'\n{"="*60}')
    print(f'📹 {video_path.name} | {format_time(total_f/fps)} | 디바이스: {DEVICE}')
    print(f'{"="*60}')

    report, prev_status, frame_idx, alert_count = [], None, 0, 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if frame_idx % frame_step != 0:
            frame_idx += 1; continue

        pil_img    = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        time_label = format_time(frame_idx / fps)

        t0      = time.time()
        status  = infer_frame(pil_img)
        elapsed = time.time() - t0

        if status != prev_status or prev_status is None:
            has_alert = ALERT_KEYWORD in status
            if has_alert: alert_count += 1
            print(f'{time_label}  {status}{"  ⚠️ 간호사 호출!" if has_alert else ""}  [{elapsed:.1f}s]')
            report.append({'time': time_label, 'status': status, 'alert': has_alert})
            prev_status = status
        else:
            print(f'{time_label}  (유지)  [{elapsed:.1f}s]')
        frame_idx += 1

    cap.release()

    # 보고서 저장
    rpath = RESULT_DIR / f'{video_path.stem}_arc_report.txt'
    with open(rpath, 'w', encoding='utf-8') as f:
        f.write(f'환자 행동 모니터링 보고서 (Intel ARC GPU)\n')
        f.write(f'생성일시: {datetime.now().strftime("%Y-%m-%d %H:%M")}\n')
        f.write('=' * 40 + '\n')
        for e in report:
            f.write(f"{e['time']}  {e['status']}{'  ⚠️' if e['alert'] else ''}\n")
        f.write(f'총 변화: {len(report)}회 | 낙상 경보: {alert_count}회\n')
    print(f'\n📄 보고서: {rpath}')